## Assistant

Here is a structured deep dive of how this repo is organized and how the poker / inference model is defined, followed by a flow chart.

---

## 1. High-level purpose

The codebase implements **Bayesian updating of beliefs about what an opponent is holding**, driven by **parametric models of how likely each action is** given (abstracted) hand strength and game state. Those action models combine:

1. A **population baseline** (multinomial logistic regression weights `β` on engineered features `φ`), optionally fit from many labeled hands (`train.py` → `artifacts/global_priors.json`).
2. A **three-dimensional tendency vector `θ`** that tilts the baseline toward more fold / passive (“call-check”) / aggressive (“raise”) play at the level of **deviations from the baseline distribution** (same mathematical pattern preflop and postflop).

Unknown hole cards are handled either by **online filtering** over 169 preflop classes or by **EM** when you want to learn `θ` from many hands where the target’s cards are hidden preflop but you observe their line.

---

## 2. Repository layout (what each layer does)

| Area | Role |
|------|------|
| **`utils/parse.py`** | Loads Pluribus `.phh` via **pokerkit**, walks actions, maintains stacks/pot, builds per-action **`State`** snapshots and **`actions`** maps per street. |
| **`utils/action_map.py`** | Maps raw actions (`f`, `cc`, `cbr`) + sizing into **discrete buckets** 0–4 (`classify`). |
| **`utils/strength/`** | **Preflop**: 169 equivalence classes, `hand_class_features`, heuristic strength. **Postflop**: best 5-card hand evaluation, draws, **`poker_hand_mapper`** (made/draw + bucket + board texture). |
| **`utils/prior/`** | **`PreflopPrior` / `PostflopPrior`**: logits = baseline `β @ φ` then **log-additive tilt** with `θ` and **behavior vectors** `u` (one-hot minus baseline probs). **`training.py`**: SGD for multiclass `β`. |
| **`utils/filter/`** | **`PreflopRangeFilter`**: Bayes filter over **169 classes**. **`PostflopStrengthFilter`**: legacy filter over **7 strength buckets** + heuristic ψ + φ (parallel story to the newer postflop feature prior). |
| **`utils/em/`** | **Preflop EM**: infer posterior over hand class, update `θ_pre`. **Postflop EM**: infer posterior over combo line, update `θ_post`. |
| **`utils/postflop_runner_bridge.py`** | Turns parsed `Hand` + known hole cards into **`PostflopFeatures`** and **`PostflopThetaObservation`** sequences for postflop EM / supervised training rows. |
| **`pipeline_common.py`** | Shared **data loading** (`flatten_hands`), **supervised row extraction** for training baselines, splits, φ column labels. |
| **`runner.py`** | Main **inference pipeline**: session/single hand → optional EM → **preflop range filter** per (observer, target) pair → JSON logs / results. |
| **`train.py`** | Fits **global** `β_preflop`, `β_facing`, `β_no_bet` from hands with known hole cards. |

Supporting **`utils/tendency.py`** only defines small datatypes (`TendencyTheta`, `ActionPrior`) that mirror the “preflop vs postflop prior” split.

---

## 3. Data: from `.phh` to decisions

**`Hand`** (`utils/parse.py`) wraps `pokerkit.HandHistory`. On **`parse()`**, it replays each raw line: blinds, hole cards (`d dh`), board (`d db`), and actions. Each actionable decision records:

- **`actions[street][t] = (player, (action_bucket, raise_level), amount)`**
- A matching **`states[street][t]`** = **`State`** with stacks, pot, condensed betting history, players alive, etc.

So every filter / EM step reads **the state before the target acts** and the **bucket** they chose.

```316:408:utils/parse.py
    def apply_action(self, raw_action):
        if not self._initialized:
            self.start_hand()
        ...
        action_bucket = classify(action_type, self._pot, bet=amount)

        self._betting_history_this_street.append((player, (action_bucket, self._street_action_level), amount))

        t = self._street_action_index[self._street]
        self.actions[self._street][t] = (player, (action_bucket, self._street_action_level), amount)
        self._street_action_index[self._street] += 1

        next_to_act = self.next_player(player)

        self._validate_stack_consistency()
        self._append_state(player_to_act=next_to_act)


    def parse(self):
        self.start_hand()
        for raw_action in self.hand_history.actions:
            self.apply_action(raw_action)
```

**`Session`** loads a **directory** of numbered `.phh` files (one hand per file).

---

## 4. Preflop action model (`PreflopPrior`)

**State abstraction**: each acting spot is summarized by **`StateKey`** — position, active player count bucket, facing bet or not, raise-count bucket, SPR bucket — built from the live **`State`** via **`state_key_from_parse_state`**.

**Hand abstraction**: the hidden variable is one of **169** string classes (`AKs`, `TT`, …). Features **`preflop_feature_vector(h, state_key)`** stack:

- Hand-class features (strength, pair/suited/gap, premium/speculative flags, …).
- One-hots for position, multiway, raise depth, SPR.

**Baseline**: three logits `β[k] @ φ` for fold / check-call / raise, passed through a floored softmax (`gto_action_probs`).

**Tendency tilt**: for each action `a`, a vector **`u_a = e_a − P_base`** (three components aligned with fold/call/raise). The tilted distribution uses **`log P_base(a) + θ_pre · u_a`** then softmax.

```429:454:utils/prior/preflop.py
    def action_probs(
        self,
        hand_class: str,
        state_key: StateKey | str,
    ) -> Dict[int, float]:
        ...
        gto_probs = self.gto_action_probs(hand_class, state_key)

        modulated_scores = {
            a: log(gto_probs[a])
            + dot3(
                self.theta_pre,
                _tendency_deviation_generic(
                    a,
                    gto_probs,
                    fold=FOLD,
                    call=CHECK_CALL,
                    raise_=RAISE,
                ),
            )
            for a in ACTION_BUCKETS
        }

        return softmax_dict_legacy(modulated_scores, floor=self.floor)
```

**Compatibility note**: legacy **`phi`** on `PreflopRangeFilter` maps to **`theta_pre ≈ (−φ, φ, φ)`** when EM is off — i.e. a single scalar looseness/tightness knob folded into the three-way tilt.

---

## 5. Preflop Bayesian filter

**`PreflopRangeFilter.update`** implements standard discrete Bayes:

\[
R_t(h) \propto R_{t-1}(h)\cdot P(a_t \mid h, s_t)
\]

with **`initial_class_prior`** = uniform over **available combos** given observer dead cards.

```56:78:utils/filter/preflop.py
    def update(
        self,
        state_key: StateKey | str,
        action_bucket: int,
    ) -> Dict[str, float]:
        ...
        unnorm: Dict[str, float] = {
            h: prob * self.prior_model.action_probability(h, state_key, action_bucket)
            for h, prob in self.range.items()
        }

        evidence = sum(unnorm.values())
        ...
        self.range = {h: v / evidence for h, v in unnorm.items()}
```

**`collapse_to_strength`** maps the 169-vector to **7 postflop buckets** by sampling a concrete combo compatible with the board and running **`poker_hand_mapper`** — this bridges preflop belief to postflop bucket beliefs if you use the strength filter path.

---

## 6. Postflop action model (`PostflopPrior`)

Here the hidden “type” is not trained end-to-end in the runner’s main line; instead **`PostflopFeatures`** summarize the spot: made/draw scores, bet fraction, pot odds, position, multiway, SPR, street, board wetness, facing bet or not.

- **Facing a bet**: 3-way softmax over fold / call / raise (`β_facing` is `3 × PHI_DIM`).
- **No bet**: call vs raise (`β_no_bet` is `2 × PHI_DIM`).

Tilting uses the same idea as preflop: baseline probs → behavior vectors → **`log p_base + θ_post · u`**.

```268:278:utils/prior/postflop.py
    def action_probs(self, features: PostflopFeatures) -> Dict[int, float]:
        legal = self.legal_actions(features)
        p_base = self.base_probs(features)
        utilities = self.action_utility_vectors(features)
        theta = self.theta_vec
        log_scores: Dict[int, float] = {}
        eps = max(self.floor, 1e-300)
        for a in legal:
            log_scores[a] = math.log(max(p_base[a], eps)) + float(theta @ utilities[a])
        probs = _softmax_log_probs(log_scores)
        return self._maybe_floor_probs(probs)
```

**`utils/postflop_runner_bridge.py`** builds **`PostflopFeatures`** from **`State`** when hole cards are known, and collapses five legacy buckets to fold/call/raise for EM rows.

---

## 7. Legacy postflop strength filter (parallel module)

**`PostflopStrengthFilter`** (`utils/filter/postflop.py`) maintains a distribution over **seven strength buckets** using a fixed heuristic ψ table and an optional scalar **`phi`** that reshapes preferences by bucket strength. This is **not** the same object as **`PostflopPrior`**, but conceptually it’s another “action likelihood × latent strength” layer.

---

## 8. EM: learning `θ` when latent variables are ambiguous

**Preflop** (`utils/em/preflop.py`):

- **E-step**: \(q(h) \propto \pi_0(h) \prod_t P(a_t \mid h, s_t)\) over 169 classes (`e_step_hand_class_posterior`).
- **M-step**: gradient ascent on **`θ_pre`** using weighted contributions from each \(q(h)\) (`m_step_theta_pre`, outer iterations **`run_preflop_em`**).

Bundles come from **`runner._collect_grouped_em_bundles`**: one bundle per hand per (observer, target) with all target preflop decisions; **`initial_range`** uses **`initial_class_prior(dead_cards=observer holes)`**.

**Postflop** (`utils/em/postflop.py`):

- When target hole cards are **known**, each hand yields one **`PostflopThetaObservation`** (combo key + list of (features, action)).
- **E-step**: posterior over combo keys (here usually one candidate or enumerations you pass in).
- **M-step**: gradient **`postflop_theta_gradient`** with **`run_postflop_theta_em`** (includes **`center_each_step`** on `θ` and clipping).

---

## 9. Training population baselines (`train.py` + `pipeline_common`)

**`train_global_priors`**:

1. **`flatten_hands`** → list of **`HandRef`**.
2. **`collect_preflop_supervised_rows`**: rows `(φ, label)` for fold/call/raise where hole cards exist.
3. **`collect_postflop_supervised_rows`**: splits into facing-bet vs no-bet datasets → **`train_baseline_facing_bet`** / **`train_baseline_no_bet`**.

Output JSON bundles **`β_preflop`**, **`β_facing`**, **`β_no_bet`** for use as **`PreflopPrior(beta_preflop=...)`** and **`PostflopPrior(beta_facing=..., beta_no_bet=...)`** (runner / EM can consume these when wired through CLI or code paths that load the artifact).

---

## 10. Runner orchestration (`runner.py`)

Conceptual sequence:

1. Load **`Session`** or single **`Hand`**.
2. Optionally **preflop EM** across the session to get **`theta_pre` per `observer|target`**.
3. Optionally **postflop EM** on hands with **known target hole cards**.
4. For each hand and each **(observer, target)** pair with target preflop actions: build **`PreflopRangeFilter`**, use **learned `PreflopPrior`** if EM ran, else **`phi`** scalar mapping, run **`update`** for each decision, store range on **`hand.hand_range[observer][target]`**, compute **`log_likelihood`** from step evidences.

```390:436:runner.py
def _run_preflop_filter_for_hand(
    hand,
    observer: str,
    target: str,
    hand_index: int,
    phi: float,
    top_k: int,
    learned_prior: Optional[PreflopPrior] = None,
) -> Optional[HandFilterResult]:
    ...
    preflop_filter = PreflopRangeFilter(
        observer_name=observer,
        target_name=target,
        observer_hole_cards=observer_hole_cards,
        prior_model=learned_prior,
    )
    if learned_prior is None:
        preflop_filter.phi = phi

    for decision in decisions:
        preflop_filter.update(decision.state_key, decision.action_bucket)

    hand.set_hand_range_vector(observer, target, preflop_filter.range)
```

---

## 11. Flow chart (end-to-end)

```mermaid
flowchart TB
  subgraph data [Data]
    PHH["Pluribus .phh files"]
    PK["pokerkit HandHistory"]
    H["Hand: states + actions per street"]
    PHH --> PK --> H
  end

  subgraph train [Offline training optional]
    ROWS["pipeline_common: supervised φ rows"]
    BETA["Fit β_preflop, β_facing, β_no_bet"]
    ART["global_priors.json"]
    ROWS --> BETA --> ART
  end

  subgraph runner [runner.py inference]
    EMpre["Preflop EM: θ_pre per observer|target"]
    EMpost["Postflop EM: θ_post when target cards known"]
    F["PreflopRangeFilter: R_t(h) updates"]
    OUT["Ranges, log L, JSON logs"]
    H --> EMpre
    H --> EMpost
    H --> F
    ART -.->|"β loaded"| F
    EMpre -->|"PreflopPrior(theta_pre)"| F
    F --> OUT
    EMpost --> OUT
  end

  subgraph models [Action likelihood kernels]
    PP["PreflopPrior: log softmax of log GTO + θ·u"]
    PF["PostflopPrior: features φ → logits + θ·u"]
    PP --- F
    PF --- EMpost
  end
```

---

## 12. Summary of the “poker model” in one sentence

The project treats betting lines as **noisy observations** of latent holdings summarized either as **169 preflop classes** or **postflop feature vectors**, with **action probabilities** given by a **softmax of population logits plus a three-way tendency correction**; **Bayesian filtering** updates beliefs when you watch a line, and **EM** estimates **player-specific `θ`** when the integration over holdings is required or when full postflop sequences with known cards are available.

If you want this drilled into a single path only (e.g. “runner only, no training” or “postflop EM only”), say which entrypoint you care about and we can trace call chains file-by-file without the parallel branches.

